# equiml — Quickstart

This notebook introduces `equiml` through two synthetic scenarios:

1. **Discrete predictions** — binary labels, hiring model
2. **Continuous predictions** — probability scores, same setting

We will show how to detect bias, read results, and use custom metrics.

In [1]:
import numpy as np
from equiml import audit
from equiml.metrics import (
    DemographicParity,
    EqualizedOdds,
    AUCParity,
    FairnessMetric,
)
from equiml.metrics.base import MetricResult

## 1. Synthetic data

We simulate a hiring model with a **controlled bias**: candidates from group 1 are systematically less likely to be hired, even when equally qualified.

In [13]:
np.random.seed(42)
n = 1000

# Sensitive attribute: 0 = group A, 1 = group B
sensitive = np.random.randint(0, 2, size=n)

# Ground truth: both groups are equally qualified
y_true = np.random.randint(0, 2, size=n)

# Biased model: group B is hired less often
# AND some false positives exist for both groups
noise = np.random.random(size=n)
y_pred_labels = y_true.copy()

# False negatives — group B misses true positives
flip_fn = (sensitive == 1) & (y_true == 1) & (noise < 0.4)
y_pred_labels[flip_fn] = 0

# False positives — some negatives predicted positive for both groups
flip_fp = (y_true == 0) & (noise < 0.15)
y_pred_labels[flip_fp] = 1

# Probas
y_pred_proba = np.where(
    sensitive == 0,
    np.clip(y_true * 0.7 + np.random.normal(0, 0.15, n), 0, 1),
    np.clip(y_true * 0.4 + np.random.normal(0, 0.15, n), 0, 1),
)

print(f"Dataset size: {n} candidates")
print(f"Group A size: {(sensitive == 0).sum()} | Group B size: {(sensitive == 1).sum()}")
print(f"Overall positive rate (labels): {y_pred_labels.mean():.2f}")
print(f"Group A positive rate: {y_pred_labels[sensitive == 0].mean():.2f}")
print(f"Group B positive rate: {y_pred_labels[sensitive == 1].mean():.2f}")

Dataset size: 1000 candidates
Group A size: 490 | Group B size: 510
Overall positive rate (labels): 0.48
Group A positive rate: 0.57
Group B positive rate: 0.39


## 2. Discrete audit

We pass binary labels to `audit()`. It automatically detects the prediction type and computes all compatible metrics.

In [14]:
result = audit(y_true, y_pred_labels, sensitive, metrics="all")
print(result)

AuditResult(pred_type='labels', groups=[np.int64(0), np.int64(1)])
Metrics:
  demographic_parity: 0.1732
  equal_opportunity: 0.3251
  equalized_odds: 0.3251
  predictive_parity: 0.0150


In [15]:
# Inspect a specific metric
dp = result.metrics["demographic_parity"]

print(f"Demographic Parity gap : {dp.value:.4f}")
print(f"Group A positive rate  : {dp.groups[0]:.4f}")
print(f"Group B positive rate  : {dp.groups[1]:.4f}")

Demographic Parity gap : 0.1732
Group A positive rate  : 0.5673
Group B positive rate  : 0.3941


In [16]:
# Equalized Odds — TPR and FPR gaps
eo = result.metrics["equalized_odds"]

print(f"Equalized Odds gap : {eo.value:.4f}")
print(f"TPR gap            : {eo.extra['tpr_gap']:.4f}")
print(f"FPR gap            : {eo.extra['fpr_gap']:.4f}")
print(f"TPR group A        : {eo.extra['tprs'][0]:.4f}")
print(f"TPR group B        : {eo.extra['tprs'][1]:.4f}")

Equalized Odds gap : 0.3251
TPR gap            : 0.3251
FPR gap            : 0.0429
TPR group A        : 1.0000
TPR group B        : 0.6749


## 3. Selecting specific metrics

In [17]:
# By name
result_partial = audit(
    y_true, y_pred_labels, sensitive,
    metrics=["demographic_parity", "equal_opportunity"]
)
print(result_partial)

AuditResult(pred_type='labels', groups=[np.int64(0), np.int64(1)])
Metrics:
  demographic_parity: 0.1732
  equal_opportunity: 0.3251


## 4. Continuous audit

We now pass probability scores. `equiml` detects the type automatically and switches to proba-compatible metrics.

In [18]:
result_proba = audit(y_true, y_pred_proba, sensitive, metrics="all")
print(result_proba)

AuditResult(pred_type='proba', groups=[np.int64(0), np.int64(1)])
Metrics:
  auc_parity: 0.0311
  calibration_parity: 0.0857
  brier_parity: 0.1278


In [19]:
# AUC Parity — per-group AUC scores
auc = result_proba.metrics["auc_parity"]

print(f"AUC Parity gap : {auc.value:.4f}")
print(f"AUC group A    : {auc.groups[0]:.4f}")
print(f"AUC group B    : {auc.groups[1]:.4f}")

AUC Parity gap : 0.0311
AUC group A    : 0.9994
AUC group B    : 0.9683


In [20]:
# Calibration Parity — ECE per group
cal = result_proba.metrics["calibration_parity"]

print(f"Calibration Parity gap : {cal.value:.4f}")
print(f"ECE group A            : {cal.groups[0]:.4f}")
print(f"ECE group B            : {cal.groups[1]:.4f}")

Calibration Parity gap : 0.0857
ECE group A            : 0.1676
ECE group B            : 0.2533


## 5. Custom metric

Any metric can be implemented by subclassing `FairnessMetric`. Here we implement a simple **Positive Rate Ratio** (also known as disparate impact ratio).

In [21]:
class DisparateImpact(FairnessMetric):
    """
    Disparate Impact Ratio.

    ratio = min(P(Ŷ=1|S=g)) / max(P(Ŷ=1|S=g))

    A ratio of 1.0 indicates no disparate impact.
    The 4/5 rule (US EEOC) flags ratios below 0.8.
    """

    @property
    def requires(self) -> set[str]:
        return {"labels"}

    @property
    def name(self) -> str:
        return "disparate_impact"

    def compute(self, y_true, y_pred, sensitive, **kwargs) -> MetricResult:
        groups = np.unique(sensitive)
        rates = {g: y_pred[sensitive == g].mean() for g in groups}
        values = list(rates.values())
        ratio = min(values) / max(values) if max(values) > 0 else 0.0
        return MetricResult(
            name=self.name,
            value=ratio,
            groups=rates,
            extra={"passes_four_fifths_rule": ratio >= 0.8},
        )


result_custom = audit(
    y_true, y_pred_labels, sensitive,
    metrics=["demographic_parity", DisparateImpact()]
)

di = result_custom.metrics["disparate_impact"]
print(f"Disparate Impact ratio   : {di.value:.4f}")
print(f"Passes 4/5 rule (>= 0.8) : {di.extra['passes_four_fifths_rule']}")

Disparate Impact ratio   : 0.6947
Passes 4/5 rule (>= 0.8) : False


## 6. Intersectional analysis

Pass a list of arrays to analyze fairness across **combinations** of sensitive attributes.

In [22]:
# Two sensitive attributes: gender and age group
gender   = np.random.randint(0, 2, size=n)
age      = np.random.randint(0, 3, size=n)  # 3 age groups

result_intersect = audit(
    y_true, y_pred_labels,
    sensitive=[gender, age],
    metrics=["demographic_parity"]
)

dp_intersect = result_intersect.metrics["demographic_parity"]
print(f"Intersectional groups: {result_intersect.groups}")
print(f"\nPositive rates per group:")
for group, rate in dp_intersect.groups.items():
    print(f"  gender={group.split('_')[0]}, age={group.split('_')[1]} → {rate:.4f}")
print(f"\nMax gap: {dp_intersect.value:.4f}")

Intersectional groups: [np.str_('0_0'), np.str_('0_1'), np.str_('0_2'), np.str_('1_0'), np.str_('1_1'), np.str_('1_2')]

Positive rates per group:
  gender=0, age=0 → 0.4479
  gender=0, age=1 → 0.5440
  gender=0, age=2 → 0.5067
  gender=1, age=0 → 0.4524
  gender=1, age=1 → 0.4085
  gender=1, age=2 → 0.5087

Max gap: 0.1354
